# 02 — Generate a 100,000-flight JSBSim skill-recognition dataset

This notebook creates a reproducible **1-v-1 F-16** dataset for neural-network training. Each aircraft is a separate JSBSim flight-dynamics instance. A seeded `StochasticSkillManager` varies geometry and skill schedules, while BVR Sim's versioned `SkillManager` supplies the commanded-skill contract and labels.

Generation is streamed one flight at a time to compressed Parquet row groups and directly to Tacview files. Memory use is therefore bounded by one flight rather than growing with the requested flight count.

> This is synthetic research data, not tactical or flight-control guidance. Open-loop control mappings are deliberately bounded and should be visually quality-checked before model training. Weapons are disabled.


## 1. Imports and reproducible build configuration

The integration timestep is finer than the logging interval. `SAMPLE_DT_S=0.1` guarantees samples every 0.1 seconds of game time; changing it to a larger value is rejected. Set `BVR_DATASET_FLIGHTS` for a shorter smoke run. `BVR_DATASET_SKILLS` is a comma-separated allow-list of tactical labels; only those skills can be sampled, so it can be changed before running this cell without editing the manager.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import subprocess
import sys
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import jsbsim
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import yaml

from bvr_behavior_prediction.data.labels import kinematic_labels
from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS
from bvr_behavior_prediction.data.schema import EPISODE_COLUMNS, TRAJECTORY_COLUMNS, validate_columns
from bvr_behavior_prediction.generation.manifest import DatasetManifest
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
BVR_SOURCE = REPO_ROOT / "bvr_sim_source"
assert BVR_SOURCE.exists(), "Expected the bundled bvr_sim_source checkout"
sys.path.insert(0, str(BVR_SOURCE))
from bvr_sim.agents.skill_manager import SkillManager

OUTPUT_DIR = Path(os.getenv(
    "BVR_NOTEBOOK_OUTPUT", REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_jsbsim_skills_v001"
))
N_FLIGHTS = int(os.getenv("BVR_DATASET_FLIGHTS", "100000"))
BASE_SEED = int(os.getenv("BVR_DATASET_SEED", "20260911"))
DURATION_S = float(os.getenv("BVR_DATASET_DURATION_S", "45"))
FLIGHTS_PER_SHARD = int(os.getenv("BVR_DATASET_FLIGHTS_PER_SHARD", "1000"))
SIM_DT_S = 1 / 60
SAMPLE_DT_S = 0.1
assert N_FLIGHTS > 0 and FLIGHTS_PER_SHARD > 0
assert 0 < SAMPLE_DT_S <= 0.1
assert round(SAMPLE_DT_S / SIM_DT_S) * SIM_DT_S == SAMPLE_DT_S
JSBSIM_ROOT = os.getenv("JSBSIM_ROOT") or None
MODEL = os.getenv("JSBSIM_MODEL", "f16")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "acmi").mkdir(exist_ok=True)
(OUTPUT_DIR / "trajectories").mkdir(exist_ok=True)
print({"flights": N_FLIGHTS, "duration_s": DURATION_S, "sample_dt_s": SAMPLE_DT_S})


In [ ]:
skills_available_for_selection = SkillManager().list_skills()
print("Skills available in SkillManager for selection:")
for skill_name in skills_available_for_selection:
    print(f"- {skill_name}")


## 2. Stochastic scenario and skill manager

Sampling is stratified over the runtime skill allow-list. Within each class, start range, lateral offset, altitude, speed, headings, switch times, and secondary skill vary. The exact schedule is retained as privileged provenance. At least two skills are required because each flight switches to a distinct secondary skill.


In [ ]:
TACTICAL_SKILLS = {
    "MAINTAIN": "maintain_heading",
    "PURSUE": "pursue_target",
    "BEAM": "beam_target_left",
    "CRANK_LEFT": "crank_target_left",
    "CRANK_RIGHT": "crank_target_right",
    "EXTEND": "extend",
}
DEFAULT_AVAILABLE_SKILLS = ("MAINTAIN", "PURSUE", "BEAM", "CRANK_LEFT", "CRANK_RIGHT", "EXTEND")
AVAILABLE_SKILLS = tuple(
    label.strip().upper()
    for label in os.getenv("BVR_DATASET_SKILLS", ",".join(DEFAULT_AVAILABLE_SKILLS)).split(",")
    if label.strip()
)
unknown_skills = set(AVAILABLE_SKILLS) - set(TACTICAL_SKILLS)
assert not unknown_skills, f"Unknown BVR_DATASET_SKILLS labels: {sorted(unknown_skills)}"
assert len(AVAILABLE_SKILLS) >= 2, "BVR_DATASET_SKILLS must contain at least two labels"
assert len(AVAILABLE_SKILLS) == len(set(AVAILABLE_SKILLS)), "BVR_DATASET_SKILLS contains duplicates"

@dataclass(frozen=True)
class FlightScenario:
    seed: int
    range_nm: float
    lateral_offset_nm: float
    observer_altitude_ft: float
    altitude_difference_m: float
    observer_speed_fps: float
    speed_difference_ms: float
    observer_heading_deg: float
    heading_difference_deg: float
    switch_time_s: float
    primary_label: str
    secondary_label: str


class StochasticSkillManager:
    """Seeded sampler restricted to an explicit BVR Sim skill allow-list."""

    def __init__(self, seed: int, available_skills):
        self.rng = random.Random(seed)
        self.available_skills = tuple(available_skills)
        self.catalogue = SkillManager()

    def sample(self, flight_index: int) -> FlightScenario:
        primary = self.available_skills[flight_index % len(self.available_skills)]
        secondary = self.rng.choice(tuple(label for label in self.available_skills if label != primary))
        return FlightScenario(
            seed=BASE_SEED + flight_index,
            range_nm=self.rng.uniform(12.0, 42.0),
            lateral_offset_nm=self.rng.uniform(-5.0, 5.0),
            observer_altitude_ft=self.rng.uniform(14_000.0, 28_000.0),
            altitude_difference_m=self.rng.uniform(-2_000.0, 2_000.0),
            observer_speed_fps=self.rng.uniform(650.0, 900.0),
            speed_difference_ms=self.rng.uniform(-60.0, 60.0),
            observer_heading_deg=self.rng.uniform(0.0, 360.0),
            heading_difference_deg=self.rng.uniform(25.0, 200.0),
            switch_time_s=self.rng.uniform(18.0, 30.0),
            primary_label=primary,
            secondary_label=secondary,
        )

    def active(self, scenario: FlightScenario, time_s: float):
        label = scenario.primary_label if time_s < scenario.switch_time_s else scenario.secondary_label
        name = TACTICAL_SKILLS[label]
        params = {"timeout_s": DURATION_S}
        if name == "maintain_heading":
            params["heading_deg"] = scenario.observer_heading_deg
        # Creating through SkillManager validates parameters and preserves its contract version.
        return label, self.catalogue.create_skill(name, params), self.catalogue.get_contract(name)

stochastic_manager = StochasticSkillManager(BASE_SEED, AVAILABLE_SKILLS)
print({"available_skills": stochastic_manager.available_skills})


## 3. JSBSim aircraft wrapper and bounded skill controls

Every aircraft receives identical model/version handling. Target control depends only on the active labelled skill and current geometry. Observer control remains a stable `MAINTAIN` reference. Skill commands are evaluated on every logged step; the controller mapping is intentionally explicit rather than treating normalized FCS commands as BVR Sim native action bins.

In [ ]:
FT_TO_M = 0.3048
FPS_TO_MPS = 0.3048
EARTH_RADIUS_M = 6_378_137.0


def clamp(value, low, high):
    return max(low, min(high, value))


def angle_error_deg(target, current):
    return (target - current + 180.0) % 360.0 - 180.0


class JSBSimAircraft:
    def __init__(self, initial):
        self.fdm = jsbsim.FGFDMExec(JSBSIM_ROOT)
        self.fdm.set_debug_level(0)
        self.fdm.set_dt(SIM_DT_S)
        if not self.fdm.load_model(MODEL):
            raise RuntimeError(f"Could not load JSBSim model {MODEL!r}")
        for key, value in initial.items():
            self.fdm[key] = value
        if not self.fdm.run_ic():
            raise RuntimeError("JSBSim rejected initial conditions")

    def state(self):
        f = self.fdm
        return {
            "latitude_deg": float(f["position/lat-gc-deg"]),
            "longitude_deg": float(f["position/long-gc-deg"]),
            "z": float(f["position/h-sl-ft"]) * FT_TO_M,
            "heading": math.radians(float(f["attitude/psi-deg"])),
            "pitch": math.radians(float(f["attitude/theta-deg"])),
            "roll": math.radians(float(f["attitude/phi-deg"])),
            "speed": float(f["velocities/vtrue-fps"]) * FPS_TO_MPS,
            "vz": float(f["velocities/v-down-fps"]) * -FPS_TO_MPS,
        }

    def step(self, controls):
        self.fdm["fcs/aileron-cmd-norm"] = controls[0]
        self.fdm["fcs/elevator-cmd-norm"] = controls[1]
        self.fdm["fcs/rudder-cmd-norm"] = controls[2]
        self.fdm["fcs/throttle-cmd-norm"] = controls[3]
        if not self.fdm.run():
            raise RuntimeError("JSBSim stopped unexpectedly")


def target_controls(label, own, opponent):
    bearing = math.degrees(math.atan2(opponent["y"] - own["y"], opponent["x"] - own["x"])) % 360
    headings = {
        "MAINTAIN": math.degrees(own["heading"]), "PURSUE": bearing,
        "BEAM": bearing - 90, "CRANK_LEFT": bearing - 45,
        "CRANK_RIGHT": bearing + 45, "EXTEND": bearing + 180,
    }
    heading_error = angle_error_deg(headings[label], math.degrees(own["heading"]))
    # Command bank, rather than holding aileron until the desired heading is crossed.
    # Roll and pitch feedback keep long manoeuvres inside the F-16 flight envelope.
    desired_roll_deg = clamp(heading_error, -35.0, 35.0)
    roll_error_deg = desired_roll_deg - math.degrees(own["roll"])
    aileron = clamp(roll_error_deg / 45.0, -0.35, 0.35)
    desired_pitch_deg = 2.0 if label in {"PURSUE", "CRANK_LEFT", "CRANK_RIGHT"} else 0.0
    elevator = clamp((math.degrees(own["pitch"]) - desired_pitch_deg) / 30.0, -0.15, 0.15)
    throttle = 0.86 if label in {"PURSUE", "EXTEND"} else 0.78
    return aileron, elevator, 0.0, throttle


def native_bins(controls):
    aileron, elevator, _rudder, throttle = controls
    return (
        int(np.clip(round(7 + 7 * aileron), 0, 14)),
        int(np.clip(round(7 - 7 * elevator), 0, 14)),
        int(np.clip(round(8 * throttle), 0, 8)), 0,
    )


## 4. Relative geometry, hierarchical labels, and Tacview writer

Coordinates use a per-flight local ENU approximation for learning while ACMI retains geodetic coordinates. Derived kinematic heads describe actual motion; `commanded_skill` and native action bins remain privileged.

In [ ]:
def add_local_state(state, lat0, lon0):
    state = dict(state)
    state["x"] = math.radians(state["longitude_deg"] - lon0) * EARTH_RADIUS_M * math.cos(math.radians(lat0))
    state["y"] = math.radians(state["latitude_deg"] - lat0) * EARTH_RADIUS_M
    horizontal = math.sqrt(max(0.0, state["speed"] ** 2 - state["vz"] ** 2))
    state["vx"] = horizontal * math.sin(state["heading"])
    state["vy"] = horizontal * math.cos(state["heading"])
    return state


def relative_features(observer, target, previous, sample_dt):
    dx, dy, dz = target["x"]-observer["x"], target["y"]-observer["y"], target["z"]-observer["z"]
    c, s = math.cos(observer["heading"]), math.sin(observer["heading"])
    dvx, dvy, dvz = target["vx"]-observer["vx"], target["vy"]-observer["vy"], target["vz"]-observer["vz"]
    rng = math.sqrt(dx*dx + dy*dy + dz*dz)
    bearing = math.atan2(dx, dy)
    los_az = math.atan2(dy, dx)
    los_el = math.atan2(dz, math.hypot(dx, dy))
    turn_rate = climb_rate = acceleration = los_rate = 0.0
    if previous:
        turn_rate = math.radians(angle_error_deg(math.degrees(target["heading"]), math.degrees(previous["heading"]))) / sample_dt
        climb_rate = (target["z"] - previous["z"]) / sample_dt
        acceleration = (target["speed"] - previous["speed"]) / sample_dt
        los_rate = angle_error_deg(math.degrees(los_az), math.degrees(previous["los_azimuth"])) * math.pi / 180 / sample_dt
    return {
        "rel_x_body": c*dx+s*dy, "rel_y_body": -s*dx+c*dy, "rel_z_body": dz,
        "rel_vx_body": c*dvx+s*dvy, "rel_vy_body": -s*dvx+c*dvy, "rel_vz_body": dvz,
        "range": rng, "range_rate": (dx*dvx+dy*dvy+dz*dvz)/max(rng, 1),
        "relative_bearing": angle_error_deg(math.degrees(bearing), math.degrees(observer["heading"])) * math.pi/180,
        "relative_heading": angle_error_deg(math.degrees(target["heading"]), math.degrees(observer["heading"])) * math.pi/180,
        "relative_altitude": dz, "relative_speed": target["speed"]-observer["speed"],
        "target_aspect": angle_error_deg(math.degrees(bearing)+180, math.degrees(target["heading"])) * math.pi/180,
        "angle_off": angle_error_deg(math.degrees(target["heading"]), math.degrees(observer["heading"])) * math.pi/180,
        "los_azimuth": los_az, "los_elevation": los_el, "los_rate": los_rate,
        "target_turn_rate": turn_rate, "target_climb_rate": climb_rate, "target_acceleration": acceleration,
    }


def write_acmi_frame(fh, time_s, observer, target):
    fh.write(f"#{time_s:.2f}\n")
    for object_id, name, color, state in ((1, "Observer", "Red", observer), (2, "Target", "Blue", target)):
        fh.write(f'{object_id},T={state["longitude_deg"]:.8f}|{state["latitude_deg"]:.8f}|{state["z"]:.2f}|{math.degrees(state["roll"]):.2f}|{math.degrees(state["pitch"]):.2f}|{math.degrees(state["heading"]):.2f},Name={name},Color={color},Type=Air+FixedWing\n')


## 5. Stream 100,000 flights

Samples are recorded at exactly 10 Hz. A fresh pair of FDM instances is constructed per flight. Rows are retained only for the current flight, converted to an Arrow table, and appended as a Parquet row group. Episode metadata is written one row at a time, while ACMI frames go straight to disk. The first sample is at game time 0.0; no wall-clock timestamps are used.


In [ ]:
def initial_conditions(scenario, target=False):
    # At this latitude, offsets are converted to geodetic initial positions.
    lat0, lon0 = 37.62, -122.38
    if target:
        north_m = scenario.range_nm * 1852.0
        east_m = scenario.lateral_offset_nm * 1852.0
        lat = lat0 + math.degrees(north_m / EARTH_RADIUS_M)
        lon = lon0 + math.degrees(east_m / (EARTH_RADIUS_M * math.cos(math.radians(lat0))))
        altitude_ft = scenario.observer_altitude_ft + scenario.altitude_difference_m / FT_TO_M
        speed_fps = scenario.observer_speed_fps + scenario.speed_difference_ms / FPS_TO_MPS
        heading = (scenario.observer_heading_deg + scenario.heading_difference_deg) % 360
    else:
        lat, lon, altitude_ft, speed_fps, heading = lat0, lon0, scenario.observer_altitude_ft, scenario.observer_speed_fps, scenario.observer_heading_deg
    return {"ic/lat-gc-deg":lat,"ic/long-gc-deg":lon,"ic/h-sl-ft":altitude_ft,"ic/psi-true-deg":heading,
            "ic/u-fps":speed_fps,"ic/v-fps":0.0,"ic/w-fps":0.0,"ic/p-rad_sec":0.0,"ic/q-rad_sec":0.0,"ic/r-rad_sec":0.0}


def run_flight(index, scenario):
    random.seed(scenario.seed); np.random.seed(scenario.seed)
    observer_fdm = JSBSimAircraft(initial_conditions(scenario, False))
    target_fdm = JSBSimAircraft(initial_conditions(scenario, True))
    episode_id = f"jsbsim-{index:06d}"
    rows = []
    previous_target = previous_rel = None
    sample_stride = round(SAMPLE_DT_S / SIM_DT_S)
    n_samples = round(DURATION_S / SAMPLE_DT_S) + 1
    skill_version = None
    acmi_path = OUTPUT_DIR / "acmi" / f"{episode_id}.txt.acmi"
    with acmi_path.open("w", encoding="utf-8-sig") as acmi:
        acmi.write("FileType=text/acmi/tacview\nFileVersion=2.1\n")
        acmi.write("0,ReferenceTime=2026-01-01T00:00:00Z,Title=JSBSim skill dataset flight\n")
        for sample_index in range(n_samples):
            time_s = sample_index * SAMPLE_DT_S
            observer_raw, target_raw = observer_fdm.state(), target_fdm.state()
            lat0, lon0 = observer_raw["latitude_deg"], observer_raw["longitude_deg"]
            observer = add_local_state(observer_raw, lat0, lon0)
            target = add_local_state(target_raw, lat0, lon0)
            label, skill, contract = stochastic_manager.active(scenario, time_s)
            skill_version = contract["version"]
            skill.execute({"time":time_s,"self_status":{"position":{"heading_deg":math.degrees(target["heading"]),"altitude_m":target["z"]},"performance":{"speed_mps":target["speed"]}}})
            controls = target_controls(label, target, observer)
            bins = native_bins(controls)
            rel = relative_features(observer, target, ({**previous_target, **previous_rel} if previous_target else None), SAMPLE_DT_S)
            lateral, vertical, energy = kinematic_labels(rel["target_turn_rate"], rel["target_climb_rate"], rel["target_acceleration"])
            row = {"episode_id":episode_id,"perspective_id":f"{episode_id}:observer","step":sample_index,"time_s":time_s,
                   "observer_id":"observer","target_id":"target","observer_aircraft_type":"F16","target_aircraft_type":"F16"}
            for state_prefix, state in (("observer",observer),("target",target)):
                row.update({f"{state_prefix}_{key}":state[key] for key in ("x","y","z","vx","vy","vz","heading","pitch","roll","speed")})
            row.update(rel)
            row.update({"track_valid":True,"track_age":0.0,"sensor_mode":"truth","target_policy":"StochasticSkillManager",
                        "target_skill":label,"target_skill_id":TACTICAL_SKILLS[label],"commanded_skill":label,
                        "target_action_heading_bin":bins[0],"target_action_altitude_bin":bins[1],"target_action_speed_bin":bins[2],"target_action_fire":bins[3],
                        "lateral_label":lateral.name,"vertical_label":vertical.name,"energy_label":energy.name,"tactical_label":label,
                        "transition_flag":int(sample_index > 0 and rows[-1]["target_skill"] != label),
                        "time_since_skill_change":time_s if time_s < scenario.switch_time_s else time_s-scenario.switch_time_s,
                        "time_to_next_skill_change":max(0.0, scenario.switch_time_s-time_s) if time_s < scenario.switch_time_s else float("inf"),
                        "target_next_skill":scenario.secondary_label if time_s < scenario.switch_time_s else label})
            for horizon in (2,4,8): row[f"transition_within_{horizon}s"] = int(0 < row["time_to_next_skill_change"] <= horizon)
            rows.append(row)
            write_acmi_frame(acmi, time_s, observer_raw, target_raw)
            previous_target, previous_rel = target, rel
            if sample_index < n_samples - 1:
                for _ in range(sample_stride):
                    observer_fdm.step(target_controls("MAINTAIN", observer, target)); target_fdm.step(controls)
    return rows, skill_version, acmi_path


class StreamingDatasetWriter:
    """Bounded-memory canonical writer; each flight becomes one Parquet row group."""

    def __init__(self, root, flights_per_shard):
        self.root = Path(root)
        self.flights_per_shard = flights_per_shard
        self.trajectory_writer = self.episode_writer = None
        self.shard_number = None

    def write_flight(self, index, trajectory_rows, episode_row):
        shard_number = index // self.flights_per_shard
        trajectory_table = pa.Table.from_pylist(trajectory_rows)
        episode_table = pa.Table.from_pylist([episode_row])
        if self.episode_writer is None:
            self.episode_writer = pq.ParquetWriter(self.root / "episodes.parquet", episode_table.schema, compression="zstd")
        if shard_number != self.shard_number:
            if self.trajectory_writer is not None:
                self.trajectory_writer.close()
            path = self.root / "trajectories" / f"shard_{shard_number:05d}.parquet"
            self.trajectory_writer = pq.ParquetWriter(path, trajectory_table.schema, compression="zstd")
            self.shard_number = shard_number
        self.trajectory_writer.write_table(trajectory_table)
        self.episode_writer.write_table(episode_table)

    def close(self):
        if self.trajectory_writer is not None:
            self.trajectory_writer.close()
        if self.episode_writer is not None:
            self.episode_writer.close()


try:
    simulator_commit = subprocess.check_output(["git","-C",str(BVR_SOURCE),"rev-parse","HEAD"],text=True).strip()
except (OSError, subprocess.CalledProcessError):
    simulator_commit = "bundled-worktree-unknown"
manifest = DatasetManifest(
    dataset_id="bvr_f16_1v1_jsbsim_skills_v001",
    simulator={"name":"JSBSim via BVR Sim SkillManager","commit":simulator_commit,"backend":"JSBSim-python","sample_dt_s":SAMPLE_DT_S,"integration_dt_s":SIM_DT_S},
    aircraft={"observer":"F16","target":"F16","model_version":MODEL},
    generation={"flight_count":N_FLIGHTS,"duration_s":DURATION_S,"base_seed":BASE_SEED,"scenario_sampling":"broad-stratified","manager":"StochasticSkillManager","available_skills":list(AVAILABLE_SKILLS),"skill_contract_version":"1.0.0"},
    fdm={"type":"JSBSim","version":getattr(jsbsim,"__version__","unknown")},
    observation={"type":"perfect-state target-centric","sensor_mode":"truth"}, weapons={"enabled":False},
)
with (OUTPUT_DIR / "manifest.yaml").open("w") as fh:
    yaml.safe_dump(manifest.as_dict(), fh, sort_keys=False)
label_map = {
    "lateral": ["STRAIGHT", "TURN_LEFT", "TURN_RIGHT"],
    "vertical": ["LEVEL", "CLIMB", "DESCEND"],
    "energy": ["STEADY_SPEED", "ACCELERATE", "DECELERATE"],
    "tactical": list(AVAILABLE_SKILLS),
}
(OUTPUT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2))
# Remove stale products from a previous run without materialising directory listings.
for pattern_dir, pattern in ((OUTPUT_DIR / "trajectories", "shard_*.parquet"), (OUTPUT_DIR / "acmi", "*.txt.acmi")):
    for stale_path in pattern_dir.glob(pattern):
        stale_path.unlink()

writer = StreamingDatasetWriter(OUTPUT_DIR, FLIGHTS_PER_SHARD)
coverage = Counter()
transition_counts = Counter()
total_samples = 0
try:
    for index in range(N_FLIGHTS):
        scenario = stochastic_manager.sample(index)
        rows, skill_version, acmi_path = run_flight(index, scenario)
        expected_samples = round(DURATION_S / SAMPLE_DT_S) + 1
        validate_columns(rows[0])
        assert len(rows) == expected_samples
        assert all(abs(row["time_s"] - step * SAMPLE_DT_S) <= 1e-9 for step, row in enumerate(rows))
        assert all(row["tactical_label"] in AVAILABLE_SKILLS for row in rows)
        assert all(np.isfinite(row[column]) for row in rows for column in MODEL_FEATURE_COLUMNS)
        with acmi_path.open(encoding="utf-8-sig") as acmi:
            assert acmi.readline().strip() == "FileType=text/acmi/tacview"
        episode_row = {"episode_id":rows[0]["episode_id"],"seed":scenario.seed,"bvr_sim_commit":simulator_commit,
            "bvr_sim_backend":"JSBSim-python","bvr_sim_config_hash":hashlib.sha256(json.dumps(asdict(scenario),sort_keys=True).encode()).hexdigest(),
            "jsbsim_version":getattr(jsbsim,"__version__","unknown"),"observer_aircraft_type":"F16","target_aircraft_type":"F16",
            "observer_model_version":MODEL,"target_model_version":MODEL,"initial_range_nm":scenario.range_nm,
            "initial_altitude_difference_m":scenario.altitude_difference_m,"initial_heading_difference_deg":scenario.heading_difference_deg,
            "initial_speed_difference_ms":scenario.speed_difference_ms,"initial_aspect_bin":"sampled-broad","scenario_bin":"broad",
            "observer_policy":"MAINTAIN","target_policy":"StochasticSkillManager","weapons_enabled":False,"sensor_mode":"truth",
            "episode_duration_s":DURATION_S,"termination_reason":"time_limit","skill_contract_version":skill_version,
            "skill_schedule_json":json.dumps({"switch_time_s":scenario.switch_time_s,"primary":scenario.primary_label,"secondary":scenario.secondary_label})}
        assert not set(EPISODE_COLUMNS) - set(episode_row)
        writer.write_flight(index, rows, episode_row)
        coverage.update(row["tactical_label"] for row in rows)
        transition_counts.update(row["tactical_label"] for row in rows if row["transition_flag"])
        total_samples += len(rows)
        del rows, episode_row
        if (index + 1) % 100 == 0:
            gc.collect()
        if (index + 1) % 1000 == 0:
            print(f"completed {index + 1}/{N_FLIGHTS}")
finally:
    writer.close()


In [ ]:
print({"trajectory_columns": len(writer.trajectory_writer.schema.names), "episode_columns": len(writer.episode_writer.schema.names)})


## 6. Validate schema, cadence, labels, leakage, and streamed artifacts

Validation happens per flight before its bounded row buffer is released. The final checks use counters and Parquet metadata rather than reconstructing the full dataset in memory. These are hard failures: do not retain a dataset that misses a flight, loses 10 Hz cadence, leaks a privileged field into the model allow-list, samples a disallowed skill, or lacks a replay.


In [ ]:
assert total_samples == N_FLIGHTS * (round(DURATION_S / SAMPLE_DT_S) + 1)
assert set(coverage) == set(AVAILABLE_SKILLS)
assert not set(MODEL_FEATURE_COLUMNS).intersection(PRIVILEGED_COLUMNS)
trajectory_files = sorted((OUTPUT_DIR / "trajectories").glob("shard_*.parquet"))
assert len(trajectory_files) == math.ceil(N_FLIGHTS / FLIGHTS_PER_SHARD)
assert sum(pq.ParquetFile(path).metadata.num_rows for path in trajectory_files) == total_samples
assert pq.ParquetFile(OUTPUT_DIR / "episodes.parquet").metadata.num_rows == N_FLIGHTS
acmi_count = sum(1 for _ in (OUTPUT_DIR / "acmi").glob("*.txt.acmi"))
assert acmi_count == N_FLIGHTS
print("Validated", N_FLIGHTS, "flights,", total_samples, "samples and", acmi_count, "Tacview files")


## 7. Inspect streamed dataset coverage

The output contains `manifest.yaml`, `episodes.parquet`, compressed trajectory shards, `label_map.json`, and `acmi/*.txt.acmi`. Labels and schedules are privileged; train only from `MODEL_FEATURE_COLUMNS`. Keep splits episode/scenario grouped. This summary has one row per allowed label, independent of dataset size.


In [ ]:
coverage_rows = [
    {"tactical_label": label, "samples": coverage[label], "transitions": transition_counts[label]}
    for label in AVAILABLE_SKILLS
]
display(coverage_rows)
print("Dataset:", OUTPUT_DIR.resolve())


## 8. Pre-training review

Before training, open several files from `acmi/` in Tacview, plot trajectories and labels around every switch, and reject unstable/crashed runs. Use episode-grouped splits, fit transforms on the training split only, and run persistence/rule baselines plus a shuffled-label leakage test. For pipeline checks, override `BVR_DATASET_FLIGHTS` with a small value before running the notebook; the production default is 100,000 flights.
